# OLS, Testing

This notebook estimates a linear regression and tests various hypotheses using standard errors assuming iid residuals (Gauss-Markov assumptions).

You may also consider the [HypothesisTests.jl](https://github.com/JuliaStats/HypothesisTests.jl) package (not used here).

## Load Packages and Extra Functions

In [1]:
MyModulePath = joinpath(pwd(),"src")
!in(MyModulePath,LOAD_PATH) && push!(LOAD_PATH,MyModulePath)
using FinEcmt_OLS

In [2]:
#=
include(joinpath(pwd(),"src","FinEcmt_OLS.jl"))
using .FinEcmt_OLS
=#

In [3]:
using DelimitedFiles, Statistics, LinearAlgebra, Distributions

## Loading Data

In [4]:
(x,header) = readdlm("Data/TwoIndustries.csv",',',header=true)

(dN,Re,F) = (x[:,1],Float64.(x[:,2:3]),Float64.(x[:,4:end]))   #make sure data is Float
x = nothing
printlnPs("Re: ",size(Re),"\n","F: ",size(F))

y = Re[:,1]                    #to get standard OLS notation, here only one return serprintlies
T = size(y,1)
x = [ones(T) F]
k = size(x,2)

println("Dependent variable: $(header[2])")
println("Regressors:")
printmat("c",header[4:end]...)
println("\nT and k: $T $k")

      Re:   (660, 2)          
       F:   (660, 3)
Dependent variable: HiTec
Regressors:
         c    Market       SMB       HML


T and k: 660 4


## OLS

(using a heteroskedasticity robust variance-covariance matrix, discussed in another notebook)

In [5]:
(b,u,_,V,R²) = OlsNW(y,x,0)
std_w = sqrt.(diag(V))

printblue("OLS Results (robust to heteroskedasticity):\n")
xNames = ["c","Market","SMB","HML"]
printmat(b,std_w;colNames=["b","std"],rowNames=xNames)

printblue("VCV*10_000 (robust to heteroskedasticity):\n")
printmat(V*10_000;colNames=xNames,rowNames=xNames)

OLS Results (robust to heteroskedasticity):

               b       std
c          0.142     0.107
Market     1.130     0.030
SMB        0.180     0.048
HML       -0.510     0.046

VCV*10_000 (robust to heteroskedasticity):

               c    Market       SMB       HML
c        114.875    -2.129    -0.095    -3.874
Market    -2.129     8.750    -5.490    -0.899
SMB       -0.095    -5.490    22.815     1.512
HML       -3.874    -0.899     1.512    20.871



## Testing a Linear Combination

Since the estimator $\hat{\beta}_{_{k\times1}}$ satisfies

$\hat{\beta}-\beta \sim N(0,V_{k\times k}),$

we can easily apply various tests.

For instance, consider

$H_0: R\beta=q,$

where $R$ is a $1 \times k$ row vector and $q$ is a scalar. In this case, we can apply a $t$-test as

$(R \hat{\beta} -q)/\omega$, 
where 
$\omega = \sqrt{R V R'}$.



In [6]:
println("The regressors are: ",xNames)
R = [0 0 1 1]                   #see xNames for the order of regressors       
q = 0
ω = sqrt(only(R*V*R'))          #only() to create scalar from 1x1 matrix
t = (only(R*b) - q)/ω

printblue("Testing b₂+b₃=0, t-stat:")
printlnPs(t)

The regressors are: ["c", "Market", "SMB", "HML"]
Testing b₂+b₃=0, t-stat:
    -4.827


## Testing a Joint Hypothesis

Consider a joint linear hypothesis of the
form

$H_0: R\beta=q,$

where $R$ is a $J \times k$ matrix and $q$ is a $J$-vector. To test this, use

$(R\beta-q)^{\prime}(RVR^{\prime}) ^{-1}(R\beta
-q)\overset{d}{\rightarrow}\chi_{J}^{2}.$

How we estimate $V$ depends on whether there is heteroskedasticity and/or autocorrelation.

In [7]:
R = [0 1 0 0;               #testing if b₂=0, b₃=0 and b₄=0
     0 0 1 0;
     0 0 0 1]
q = [0,0,0]
test_stat = (R*b-q)'inv(R*V*R')*(R*b-q)    #R*V*R' is 2x2

printblue("Testing Rb = [0,0,0] (χ²):")
printmat([test_stat,quantile(Chisq(2),0.9)];rowNames=["test statistic","10% critical value"])

Testing Rb = [0,0,0] (χ²):
test statistic      1959.030
10% critical value     4.605



## Regression Diagnostics: Testing All Slope Coefficients

The `OlsR2Test()` function tests all slope coefficients (or equivalently, the $R^2$) of a regression, under the assumption of *iid residuals*. Notice that the regression must contain an intercept for R² to be useful. The test is to previous joint test, except that the current test assumes iid residuals, so the results can differ.

In [8]:
@doc2 OlsR2Test

```julia
OlsR2Test(R²,T,df)
```

Test of all slope coefficients. Notice that the regression must contain an intercept for R² to be useful.

### Input

  * `R²::Number`:    R² value
  * `T::Int`:        number of observations
  * `df::Number`:    number of (non-constant) regressors

### Output

  * `RegrStat::Number`: test statistic
  * `pval::Number`:     p-value


In [9]:
using CodeTracking
println(@code_string OlsR2Test(1.0,1,25))    #print the source code

function OlsR2Test(R²,T,df)
    RegrStat = T*R²/(1-R²)           #R\^2[TAB]
    pval     = ccdf(Chisq(df),RegrStat)    #same as 1-cdf()
    return RegrStat, pval
end


In [10]:
df = size(x,2) - 1              #number of slope coefficients
(RegrStat,pval) = OlsR2Test(R²,T,df)

printblue("Test of all slopes = 0:\n")
printmat([RegrStat,pval],rowNames=["stat","p-val"])

println("The result looks more significant than before, since it assumes iid")

Test of all slopes = 0:

stat   3072.554
p-val     0.000

The result looks more significant than before, since it assumes iid
